# Gene Family Classification — Guided Exercises

**AMLG | DNA sequence representation, k-mer features, and supervised classification**

In this notebook, you will turn DNA sequences into numerical features and train a simple machine-learning model to predict **gene family labels**. We begin with small examples before applying the same steps to gene sequences from humans, chimpanzees, and dogs. The original exercises and material are retained, with additional explanations and optional commented solutions.

## How to use this notebook

1. Run cells **top to bottom** using **Shift + Enter**.
2. For each **❓ exercise**, read **Task**, **Hints**, **What this teaches**, and **Expected outcome**.
3. Complete the `# TODO` markers. The **commented code under `# --- Solution ---`** is an optional answer key: uncomment the relevant lines if you are stuck or during a classroom demo.
4. Compare your results with the expected *type, structure, or behavior*. Numerical model scores are not fixed and may vary.
5. The existing file paths (`data/chimpanzee.txt`, `data/dog.txt`, `data/human.txt`) and download setup are intentionally **unchanged**. Run this notebook from the same working directory as the original.

**Learning route:** DNA strings → NumPy arrays → ordinal and one-hot encodings → k-mers → Bag of Words → read the gene-family dataset → create features → train/test split → classifier → evaluation → optional improvements.

> **Learning goals:** Explain why models need numerical features; compare three representations; interpret a k-mer count matrix; distinguish features (`X`) from labels (`y`); and evaluate a multi-class classifier. In the final dataset, gene-family labels are already provided: this is **supervised classification**, not discovery of new families.


# **Classification of Gene Sequences Into Their Gene Families**

First, let's download the data for this notebook.

In [ ]:
import os
import sys

# Append the root of the Git repository to the path.
git_root = os.popen(cmd="git rev-parse --show-toplevel").read().strip()
sys.path.append(git_root)

In [ ]:
from utils import download_file

if not os.path.exists(path="data"):
    os.makedirs(name="data")

download_file(
    # url="https://dataverse.harvard.edu/api/access/datafile/10493562",
    url="https://seafile.cloud.uni-hannover.de/d/5d6029c6eaaf410c8b01/files/?p=%2Fgene_family_classification%2Fchimpanzee.txt&dl=1",
    save_filename="data/chimpanzee.txt",
)
download_file(
    # url="https://dataverse.harvard.edu/api/access/datafile/10493561",
    url="https://seafile.cloud.uni-hannover.de/d/5d6029c6eaaf410c8b01/files/?p=%2Fgene_family_classification%2Fdog.txt&dl=1",
    save_filename="data/dog.txt",
)
download_file(
    # url="https://dataverse.harvard.edu/api/access/datafile/10493560",
    url="https://seafile.cloud.uni-hannover.de/d/5d6029c6eaaf410c8b01/files/?p=%2Fgene_family_classification%2Fhuman.txt&dl=1",
    save_filename="data/human.txt",
)

Second, let's group some imports here.

In [ ]:
import re
from typing import Any, List

import numpy as np
import numpy.typing as npt

**Concept check:** A DNA sequence is a string such as `ACGT`. A classifier cannot use raw characters directly as ordinary numerical features. We will compare ways to represent the same sequence numerically. Here, `N` means an unknown or unsupported base.

# Biological Sequence Representation for Machine Learning

Machine learning models generally require input to be numerical values organized in the form of feature matrices.
Biological sequence data is however usually provided in a string-like format.

There are three basic approaches to represent (or in other words, encode) biological sequence data:

1. Ordinal encoding
2. One-hot encoding
3. Biological sequences as a "language" through _k_-mer counting

Before we start, however, we need a function that harmonizes biological sequence strings and converts them into more convenient `numpy` arrays:

### Guided exercise · step 1

**Task:** Normalize DNA sequence text and reshape it into an array.

**Hints:** Try `sequence.upper()` and `re.sub("[^ACGT]", "N", sequence)` after uppercasing. NumPy expects one character per row.

**Expected outcome:** Input `"gaT!aca"` becomes seven uppercase rows, with `!` represented by `N`. `"GATTACA"` prints a `(1, 7)` row when transposed.

**What this teaches:** Data cleaning prevents lowercase letters or unexpected symbols from becoming different categories.

**Check yourself:** Can you describe the output in one sentence before running the next cell?

##### ❓ Converting biological sequence strings to `numpy` arrays

Complete the following function to harmonize (lowercase to uppercase, use `ACGTN` only) the biological sequence strings.

In [ ]:
def sequence_to_array(sequence: str) -> npt.NDArray[Any]:
    """Convert a DNA sequence to a NumPy array."""
    # TODO: uppercase the sequence, then replace unsupported bases with N.
    # --- Solution ---
    # sequence = sequence.upper()
    # sequence = re.sub(pattern="[^ACGT]", repl="N", string=sequence)


    # Format our output array.
    array = np.array(list(sequence))
    array = np.reshape(array, newshape=(len(array), 1))

    return array

In [ ]:
print(sequence_to_array(sequence="GATTACA").T)

**Important distinction:** Ordinal encoding assigns numbers to categories, but the numeric distance between bases has **no biological meaning**. For example, an encoding of `T` as 4 does not make it “twice” `C` as 2.

## Ordinal Encoding

In this approach, we encode bases (i.e., categorical features) as ordinal integers, i.e., integers having a relative ordering.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

ordinal_encoder = OrdinalEncoder()
ordinal_encoder.fit(X=np.array([["A"], ["C"], ["G"], ["T"], ["N"]]))

array = sequence_to_array(sequence="GATTACAN")
encoded_sequence = ordinal_encoder.transform(X=array)

with np.printoptions(
    formatter={"float_kind": lambda x: f"{x:3.0f}", "str_kind": lambda x: f"  {x}"}
):
    print(f"Sequence:         {array.T}")
    print(f"Encoded sequence: {encoded_sequence.T}")

## One-Hot Encoding

Another approach is to use one-hot encoding to represent the biological sequence.
This is widely used in deep learning methods.
Each symbol gets transformed into a one-hot encoding vector.
A one-hot encoding vector is a vector in which all elements are zero except for one element which is set to one.
The position of the one in the vector encodes the symbol.
For further processing, these one-hot encoded vectors can for example be concatenated.

### Guided exercise · step 2

**Task:** Encode the sequence `GATTACA` as one-hot rows.

**Hints:** Import `OneHotEncoder`, create `OneHotEncoder(sparse_output=False)`, then `fit_transform` the `sequence_to_array` output. Each row represents one base.

**Expected outcome:** An array with **7 rows** and one column per observed base; each row has exactly one `1`. The set of columns depends on fitted categories.

**What this teaches:** One-hot encoding does not imply a numerical ranking of bases.

**Check yourself:** Can you describe the output in one sentence before running the next cell?

##### ❓ One-hot encoding for biological sequences

Similar to the ordinal encoding, implement and perform the one-hot encoding for the sequence `GATTACA`.
Use the `OneHotEncoder` class from `sklearn.preprocessing`.
When instantiating an encoder from the class, use `sparse_output=False`.

In [ ]:
# TODO: initialize a one-hot encoder and encode the DNA array.
# --- Solution ---
# from sklearn.preprocessing import OneHotEncoder
# array = sequence_to_array(sequence="GATTACA")
# one_hot_encoder = OneHotEncoder(sparse_output=False)
# encoded_sequence = one_hot_encoder.fit_transform(array)


with np.printoptions(
    formatter={"float_kind": lambda x: f"{x:3.0f}", "str_kind": lambda x: f"  {x}"}
):
    print(f"{array.T}")
    print(f"{encoded_sequence.T}")  # type: ignore

**Small example:** For `ACGTA` and `k=3`, overlapping words are `ACG`, `CGT`, and `GTA`. K-mer counts can be fixed-width even when original sequences have different lengths, though plain counts lose the original order.

## Biological Sequences as a "Language" Through K-Mer Counting

A hurdle that still remains is that none of these above methods results in arrays of uniform length, which is often a necessity.
So with the above methods, you have to resort to things like truncating sequences or padding with "N" to get arrays of uniform length.

Luckily there is the concept of _k_-mers.

_k_-mers are substrings of length $k$ contained within a biological sequence.
They are a fairly simple concept that turns out to be tremendously powerful.

Usually, the term _k_-mer refers to all of a sequence's subsequences of length $k$.
Using this definition, the sequence `ACGT` would have four monomers (`A`, `C`, `G`, `T`), three 2-mers (`AC`, `CG`, `GT`), two 3-mers (`ACG`, `CGT`), and one 4-mer (`ACGT`).
More generally, a sequence of length $L$ has $L - k + 1$ _k_-mers and $n^k$ total possible _k_-mers, where $n$ is the number of possible monomers.

In some sense, by using _k_-mers, we can also incorporate the concepts of "words" (_k_-mers) and "sentences" (sequences of _k_-mers) from natural language processing.

The following function `build_kmers()` returns a list of _k_-mer "words" for a given sequence.

In [ ]:
def build_kmers(sequence: str, k: int) -> List[str]:
    """Build k-mers from a sequence."""
    return [sequence[i : i + k] for i in range(len(sequence) - k + 1)]

We can use this function to generate a language-like sentence from a biological sequence:

In [ ]:
sequence = "GTGCCCAGGTTCAGTGA"
words = build_kmers(sequence=sequence, k=3)
sentence = " ".join(words)

print(f"Sequence : {sequence}")
print(f"Words    : {words}")
print(f"Sentence : {sentence}")

In a next step, we may use a **Bag of Words** (BoW) model to represent our "sentence".

> A Bag of Words (BoW) model is a popular and simple way to represent text data in natural language processing (NLP) and information retrieval.
> In this model, a document is represented as an unordered set of words, disregarding grammar and word order but keeping track of word frequency.
> The name "Bag of Words" implies that the model is only concerned with the occurrence of words in a document, not with their order or structure.

> Here's how the Bag of Words model typically works:
>
> 1. Tokenization: The first step is to break down a text into individual words or tokens. This process involves removing punctuation, converting all words to lowercase, and splitting the text into individual words.
> 2. Vocabulary Building: Create a vocabulary of all unique words in the entire corpus (collection of documents). Each unique word in this vocabulary is assigned a unique index.
> 3. Vectorization: Represent each document as a vector in the space of the vocabulary. The vector has the same length as the vocabulary, and each element of the vector corresponds to the frequency of a word from the vocabulary in the document.

> Here's a simple example:
>
> Consider two documents:
>
> - Document 1: "The cat in the hat."
> - Document 2: "The quick brown fox."
> - The vocabulary is: {the, cat, in, hat, quick, brown, fox}.
>
> The Bag of Words representation for these documents would be:
>
> - Document 1: [2, 1, 1, 1, 0, 0, 0]
> - Document 2: [1, 0, 1, 0, 1, 1, 1]

### Guided exercise · step 3

**Task:** Represent multiple sequences as k-mer count features.

**Hints:** For each DNA string call `build_kmers(sequence, 3)`, join with spaces, and use `CountVectorizer` with the resulting sentences. `fit_transform` builds the vocabulary and matrix.

**Expected outcome:** Three rows (one per sequence), columns for observed 3-mers, nonnegative integer counts. The vocabulary can be inspected using `get_feature_names_out()`.

**What this teaches:** A fixed-width count vector allows differently sized DNA strings to be compared.

**Check yourself:** Can you describe the output in one sentence before running the next cell?

##### ❓ Employing the BoW model in the context of biological sequence data

The BoW model can&nbsp;&mdash;&nbsp;analogous to its use in NLP&nbsp;&mdash;&nbsp;be employed for biological sequence data.

1. Use the function `build_kmers()` to build a word corpus from the given sequence corpus. Use $k=3$.
2. Use the `CountVectorizer` from `sklearn.feature_extraction.text` to fit the BoW model to the word corpus.
3. Vectorize new "documents" using the trained BoW model.

> Use the function `toarray()` to convert sparse matrix objects returned by `CountVectorizer`s `transform()` and `fit_transform()` functions into dense representation as a NumPy array.

In [ ]:
sequence_corpus = ["GATTCCAG", "GATTACA", "NAAGT"]

In [ ]:
# Build the word corpus.

# TODO: Complete this exercise.
# --- Solution ---
# word_corpus = [" ".join(build_kmers(seq, k=3)) for seq in sequence_corpus]


In [ ]:
# Fit the BoW model to the word corpus.

from sklearn.feature_extraction.text import CountVectorizer

# TODO: Complete this exercise.
# --- Solution ---
# vectorizer = CountVectorizer()
# word_vectorization = vectorizer.fit_transform(word_corpus)


print(f"Word vectorization: {word_vectorization.toarray()}")  # type: ignore
print(f"Feature names: {vectorizer.get_feature_names_out()}")

In [ ]:
# Vectorize new documents using the trained BoW model.

# TODO: Complete this exercise.
# --- Solution ---
# new_sequences = ["GATTACA", "ACGTAC"]
# new_corpus = [" ".join(build_kmers(seq, k=3)) for seq in new_sequences]
# new_vectorizations = vectorizer.transform(new_corpus)
# print(new_vectorizations.toarray())


## From toy example to supervised prediction

The preceding exercises teach *feature extraction*. Now each row becomes an observation, the DNA sequence becomes input **X**, and the known gene-family ID becomes target **y**. Although we load three species, the original modeling workflow trains and evaluates on **human sequences only**; chimpanzee and dog tables are loaded for potential extensions.

# A Real-World End-to-End Exercise

In this end-to-end exercise, the goal is to classify gene sequences into their gene families.

We provide gene sequences from three species in the `data/` folder:

- [Chimpanzee](https://en.wikipedia.org/wiki/Chimpanzee) (_Pan troglodytes_): `chimpanzee.txt`
- [Dog](https://en.wikipedia.org/wiki/Dog) (_Canis familiaris_): `dog.txt`
- [Human](https://en.wikipedia.org/wiki/Human) (_Homo sapiens_): `human.txt`

Each file contains two tab-separated columns.
The first column (column header: `sequence`) contains the relevant gene sequence.
The second column (column header: `class`) contains the corresponding gene family, coded as integer:

| Identifier | Gene family                         |
|------------|-------------------------------------|
|          0 | G protein-coupled receptors (GPCRs) |
|          1 | Tyrosine kinase                     |
|          2 | Tyrosine phosphatase                |
|          3 | Synthetase                          |
|          4 | Synthase                            |
|          5 | Ion channel                         |
|          6 | Transcription factor                |

### Guided exercise · step 4

**Task:** Load the three labeled gene-sequence tables.

**Hints:** Use `pd.read_table("data/human.txt")` and the analogous paths. Inspect `.head()`, `.shape`, and `.columns`.

**Expected outcome:** Three pandas DataFrames. The human table should have `sequence` and `class` columns.

**What this teaches:** Tabular data lets us keep each input sequence paired with its known target label.

**Check yourself:** Can you describe the output in one sentence before running the next cell?

##### ❓ Reading data in with `pandas`

Use `pandas`' [`read_table()`](https://pandas.pydata.org/docs/reference/api/pandas.read_table.html) function to read in the three files.

In [ ]:
import pandas as pd

# TODO: Replace None with the three data-loading statements.
chimp_dna = None
dog_dna = None
human_dna = None

# --- Solution ---
# chimp_dna = pd.read_table("data/chimpanzee.txt")
# dog_dna = pd.read_table("data/dog.txt")
# human_dna = pd.read_table("data/human.txt")


In [ ]:
human_dna.head()

### Guided exercise · step 5

**Task:** Generate overlapping 6-mers for human sequences.

**Hints:** Use `.apply(lambda seq: build_kmers(seq, k=6))` on the `sequence` column; store result in a new `words` column.

**Expected outcome:** Each row now includes a list of 6-character words. A sequence of length L (L ≥ 6) produces L − 5 words.

**What this teaches:** K-mer tokenization turns biological strings into reusable local patterns.

**Check yourself:** Can you describe the output in one sentence before running the next cell?

##### ❓ Building _k_-mer "words" for the data

Convert the sequences into short overlapping _k_-mer "words" of length $k=6$.
For now, do this only for the human data.
Add the _k_-mer words as additional column to the `human_dna` dataframe.

In [ ]:
# TODO: Complete this exercise.
# --- Solution ---
# human_dna["words"] = human_dna["sequence"].apply(lambda seq: build_kmers(seq, k=6))
# human_dna[["sequence", "words"]].head()


### Guided exercise · step 6

**Task:** Convert each list of 6-mers into a sentence.

**Hints:** Use `" ".join(words)` once for each row in `human_dna["words"]`.

**Expected outcome:** `human_corpus` is a list of strings separated by spaces; its length equals the number of human sequences.

**What this teaches:** The vectorizer treats k-mers as text tokens.

**Check yourself:** Can you describe the output in one sentence before running the next cell?

##### ❓ Building "sentences"

For the human data, convert the lists of _k_-mers into sentences of words (separated by one whitespace) that can be used to fit the BoW model.

In [ ]:
# TODO: Convert lists of words into a corpus of space-separated strings.
human_corpus = None

# --- Solution ---
# human_corpus = human_dna["words"].apply(lambda words: " ".join(words)).tolist()


### Guided exercise · step 7

**Task:** Build the human k-mer feature matrix.

**Hints:** Use `CountVectorizer(ngram_range=(1, 1))` and `fit_transform(human_corpus)`. Be mindful that the resulting matrix is sparse.

**Expected outcome:** `human_vectorizations.shape` reports `(number_of_sequences, number_of_features)`.

**What this teaches:** Each sequence becomes a feature vector of k-mer frequencies.

**Check yourself:** Can you describe the output in one sentence before running the next cell?

##### ❓ Fit a BoW model

Use the `CountVectorizer` from `sklearn.feature_extraction.text` to fit a BoW model to the `human_corpus`.
Also, vectorize the entire corpus.

> Play around with the `ngram_range` argument of `CountVectorizer()` to introduce more context into the vectorizations.

In [ ]:
# TODO: Initialize and fit the vectorizer.
vectorizer = None
human_vectorizations = None

# --- Solution ---
# vectorizer = CountVectorizer(ngram_range=(1, 1))
# human_vectorizations = vectorizer.fit_transform(human_corpus)

print(f"Vectorizations shape: {human_vectorizations.shape}")


### Guided exercise · step 8

**Task:** Split features and labels into train and test sets.

**Hints:** Extract `human_dna["class"]` into `y`. Import `train_test_split`; use `test_size=0.2`, `random_state=42`, and optionally `stratify=y` if every class supports it.

**Expected outcome:** `X_train`, `X_test`, `y_train`, and `y_test` exist; about 80% of rows are training examples.

**What this teaches:** Testing on held-out examples gives a more realistic view of prediction performance.

**Check yourself:** Can you describe the output in one sentence before running the next cell?

> **Methodology note (optional):** The original notebook fits the vocabulary to all human sequences *before* train/test splitting. This is convenient for teaching, but for a stricter evaluation split the raw sequences first and fit `CountVectorizer` on the **training** corpus only; then call `transform` on the test corpus. Do not use test labels when fitting or selecting your model.

##### ❓ Split the data into a training and a test set

Extract the class labels from the `human_dna` dataframe.
Then, use the function [`train_test_split()`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html) from `sklearn.model_selection` to split the vectorizations _and_ the class labels into a training set (80%) and a test set (20%).

In [ ]:
# TODO: Complete this exercise.
# --- Solution ---
# from sklearn.model_selection import train_test_split
# y = human_dna["class"]
# X_train, X_test, y_train, y_test = train_test_split(
#     human_vectorizations, y, test_size=0.2, random_state=42, stratify=y
# )


### Guided exercise · step 9

**Task:** Train a baseline multi-class classifier and predict labels.

**Hints:** Start with `MultinomialNB()` because the input contains counts. Fit on `X_train, y_train`, then call `.predict(X_test)`.

**Expected outcome:** `y_pred` contains one predicted family ID for each test sequence; the following confusion matrix and metrics cells run.

**What this teaches:** Supervised learning maps observed k-mer patterns to known gene-family classes.

**Check yourself:** Can you describe the output in one sentence before running the next cell?

##### ❓ Train a classifier

Take a classifier of your choice to classify the data.

> Try for example:
>
> - `sklearn.naive_bayes.MultinomialNB`
> - `sklearn.tree.DecisionTreeClassifier`
> - `sklearn.ensemble.RandomForestClassifier`

In [ ]:
# TODO: Complete this exercise.
# --- Solution ---
# from sklearn.naive_bayes import MultinomialNB
# classifier = MultinomialNB()
# classifier.fit(X_train, y_train)
# y_pred = classifier.predict(X_test)


**Reading the results:** A **confusion matrix** compares true labels (rows) and predicted labels (columns). Most predictions should ideally lie on the diagonal. **Accuracy** is the fraction correct. **Weighted F1** accounts for per-class precision and recall while weighting classes by their size; it may hide weak performance on rare classes.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

ConfusionMatrixDisplay.from_predictions(y_true=y_test, y_pred=y_pred)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

print(f"Accuracy  : {accuracy_score(y_true=y_test, y_pred=y_pred):.4f}")
print(
    f"Precision : "
    f"{precision_score(y_true=y_test, y_pred=y_pred, average='weighted'):.4f}"
)
print(
    f"Recall    : {recall_score(y_true=y_test, y_pred=y_pred, average='weighted'):.4f}"
)
print(f"F1 score  : {f1_score(y_true=y_test, y_pred=y_pred, average='weighted'):.4f}")

## Reflection and optional stretch goals

Try answering before changing any code: (1) Why are k-mer count vectors the same width for different sequence lengths? (2) What order information does Bag of Words discard? (3) Does high accuracy imply every family is recognized equally well? (4) What would change if you tested predictions on chimpanzee or dog sequences?

**Optional:** Compare weighted F1 with `average="macro"`, inspect class counts, and repeat evaluation with different seeds. For rigorous evaluation, split the sequences *before* fitting the vectorizer.

# Enhancements

- Try to use different `max_depth`s for the `DecisionTreeClassifier`.
- The classes are imbalanced. Try to upsample the minority classes using `resample()` from `sklearn.utils`. Also try to downsample the majority class(es).
- It could be beneficial to tune some hyperparameters (e.g., `CountVectorizer`'s `ngram_range` parameter and the _k_-mer length).